# Lithuanian speech-to-text — Colab quickstart

Transcribes Lithuanian audio to a punctuated, speaker-labelled transcript
and subtitle files, using [`paprika-whisper-lt-v3`](https://huggingface.co/kristijonas/paprika-whisper-lt-v3).

**Set the runtime to GPU first**: Runtime → Change runtime type → T4 GPU.
It works on CPU, but a 10-minute recording takes roughly 20 minutes instead
of about one.

Repo: https://github.com/kristijonasatpro/paprika-lt-asr

## 1. Install

Takes 2–3 minutes. Colab already has torch and ffmpeg.

In [ ]:
!git clone -q https://github.com/kristijonasatpro/paprika-lt-asr
%cd paprika-lt-asr
!pip install -q transformers soundfile librosa punctuators onnxruntime sherpa-onnx

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE — set Runtime > Change runtime type > T4 GPU')

## 2. Add your audio

Run the cell and pick a file (m4a, mp3, wav, mp4, mov — anything ffmpeg
reads). Or skip it and mount Drive instead:

```python
from google.colab import drive; drive.mount('/content/drive')
AUDIO = '/content/drive/MyDrive/your-recording.m4a'
```

In [ ]:
from google.colab import files
up = files.upload()
AUDIO = list(up)[0]
print('using', AUDIO)

## 3. Transcribe

Downloads ~1.6 GB of weights on the first run.

Pass `--speakers N` when you know how many people are talking — it is more
reliable than letting a threshold guess. Use `--no-diar` to skip speaker
labels entirely (faster), or `--no-punct` for raw lowercase output.

In [ ]:
!python transcribe_file.py "$AUDIO" --speakers 2

## 4. Read it, then download

In [ ]:
import pathlib
stem = pathlib.Path(AUDIO).with_suffix('')
print(pathlib.Path(f'{stem}.txt').read_text()[:3000])

In [ ]:
for ext in ('txt', 'srt', 'vtt', 'json'):
    p = pathlib.Path(f'{stem}.{ext}')
    if p.exists():
        files.download(str(p))

## Using the model directly

If you only want the ASR and not the pipeline, this is the whole thing.

**Do not use `chunk_length_s`.** The chunked pipeline decodes fixed windows
independently and merges them by matching text in the overlaps — where the
two sides disagree it discards the span. On clean audio we measured it
silently dropping 30 and 52 words at seams, with nothing in the output to
mark the gap. It also invents 24–164 characters of Lithuanian when given
silence or room tone, where the code below emits nothing.

Native long-form holds the whole feature sequence in memory (~18 GB for 22
minutes), so for long files use `transcribe_file.py` above — it cuts into
pause-aligned blocks and stays flat at about 5.4 GB whatever the duration.

In [ ]:
import torch, numpy as np
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transcribe_file import load_audio

M = 'kristijonas/paprika-whisper-lt-v3'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
dt = torch.float16 if dev == 'cuda' else torch.float32
proc = WhisperProcessor.from_pretrained(M, language='lithuanian', task='transcribe')
mdl = WhisperForConditionalGeneration.from_pretrained(M, dtype=dt).to(dev).eval()

audio = load_audio(AUDIO)[:30 * 16000]   # first 30 seconds
f = proc(audio, sampling_rate=16000, return_tensors='pt',
         truncation=False, padding='longest', return_attention_mask=True)
with torch.no_grad():
    ids = mdl.generate(f.input_features.to(dev, dt),
                       attention_mask=f.attention_mask.to(dev),
                       language='lithuanian', task='transcribe',
                       return_timestamps=True, condition_on_prev_tokens=False,
                       temperature=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
                       logprob_threshold=-1.0,
                       compression_ratio_threshold=1.35,
                       no_speech_threshold=0.6)
print(proc.batch_decode(ids, skip_special_tokens=True)[0].strip())

Output is lowercase and unpunctuated by design — the training labels are,
and a dedicated tagger does the job better:

```python
from punct_restore import Punctuator
text, stats = Punctuator().restore(raw_text)
```

It changes case and punctuation only; if the word sequence would change at
all, it returns the input untouched and counts the refusal.

---

**Live subtitles** (`subtitle_stream.py`) need a microphone, so they do not
work in Colab — run that one locally.